In [1]:
from physics.simulation import mcfm, msq
from physics.hzz import zz4l
from physics.hstar import c6

import numpy as np
import json
import hist
import matplotlib, matplotlib.pyplot as plt


In [2]:
matplotlib.use("pgf")
matplotlib.rcParams.update({
    "pgf.texsystem": "lualatex",
    "text.usetex": True,
    "pgf.rcfonts": False,
    "pgf.preamble": "", 
})
lw  = 1.0


In [ ]:
with open('data/zz4l/xsec.json') as f:
    xsec = json.load(f)

xsec = {
    msq.Component.SBI : xsec['gg4l_sbi'] * 1.83 * 2,
    msq.Component.SIG : xsec['gg4l_sig'] * 1.83 * 2,
    msq.Component.INT : xsec['gg4l_int'] * 1.83 * 2,
    msq.Component.BKG : xsec['gg4l_bkg'] * 1.83 * 2
}

filenames = {
    msq.Component.SBI : 'data/eft/zz4l/ggZZ_sbi/events.csv',
    msq.Component.SIG : 'data/eft/zz4l/ggZZ_sig/events.csv',
    msq.Component.INT : 'data/eft/zz4l/ggZZ_int/events.csv',
    msq.Component.BKG : 'data/eft/zz4l/ggZZ_bkg/events.csv'
}

component_names = {
    msq.Component.SBI : 'SBI',
    msq.Component.SIG : 'SIG',
    msq.Component.INT : 'INT',
    msq.Component.BKG : 'BKG'
}

In [4]:
events_sig = mcfm.from_csv(cross_section=xsec[msq.Component.SIG], file_path = filenames[msq.Component.SIG], n_rows=3_000_000)
events_sig = zz4l.analyze(events_sig)

events_sbi = mcfm.from_csv(cross_section=xsec[msq.Component.SBI], file_path=filenames[msq.Component.SBI], n_rows=3_000_000)
events_sbi = zz4l.analyze(events_sbi)

In [5]:
c6_vals = [20, -20]
c6_colors = ['red', 'blue']
c6_lines = ['-', '-']

mod_sig = c6.Modifier(baseline=msq.Component.SIG, events=events_sig, c6_values=[-20,-10,0,10,20])
wt_sig_c6, prob_sig_c6 = mod_sig.modify(c6_vals)

mod_sbi = c6.Modifier(baseline=msq.Component.SBI, events=events_sbi, c6_values=[-20,-10,0,10,20])
wt_sbi_c6, prob_sbi_c6 = mod_sbi.modify(c6_vals)

In [38]:
color_sm = 'black'
line_sm = '--'

In [39]:
xobs_sbi = events_sbi.kinematics['4l_mass'].to_numpy()
xobs_sig = events_sig.kinematics['4l_mass'].to_numpy()

nxbins = 41
xmin, xmax = 180, 1000.0
xbins = np.linspace(xmin, xmax, nxbins + 1)
xcenters = 0.5 * (xbins[:-1] + xbins[1:])
xwidths = np.diff(xbins)

In [40]:
fb_to_ab = 1000.0
h_sig_sm = hist.Hist(hist.axis.Variable(xbins), storage=hist.storage.Weight())
h_sig_sm.fill(xobs_sig, weight=events_sig.weights * fb_to_ab)

h_sig_c6 = []
for i_c6, c6_val in enumerate(c6_vals):
    h = hist.Hist(hist.axis.Variable(xbins), storage=hist.storage.Weight())
    h.fill(xobs_sig, weight=wt_sig_c6[:,i_c6] * fb_to_ab)
    h_sig_c6.append(h)

h_sbi_sm = hist.Hist(hist.axis.Variable(xbins), storage=hist.storage.Weight())
h_sbi_sm.fill(xobs_sbi, weight=events_sbi.weights)

h_sbi_c6 = []
for i_c6, c6_val in enumerate(c6_vals):
    h = hist.Hist(hist.axis.Variable(xbins), storage=hist.storage.Weight())
    h.fill(xobs_sbi, weight=wt_sbi_c6[:,i_c6])
    h_sbi_c6.append(h)


In [41]:
fig, (ax1, ax2) = plt.subplots(2,1, gridspec_kw={'height_ratios': [3,1]}, figsize=(5,5), sharex=True)

l_c6 = []
for i_c6, c6_val in enumerate(c6_vals):
    l_c6.append(ax1.stairs(h_sbi_c6[i_c6].values(), xbins, color=c6_colors[i_c6], linestyle=c6_lines[i_c6], linewidth=lw))
l_sm = ax1.stairs(h_sbi_sm.values(), xbins, color=color_sm, linestyle=line_sm, linewidth=lw)

l_sm.set_label('$\mathrm{SM}$')
for i_c6, c6_val in enumerate(c6_vals):
    sgn = '-' if c6_val < 0 else '+'
    l_c6[i_c6].set_label(f'$C_{{H}} = {sgn}{abs(c6_val):d}$')
ax1.legend(frameon=False, fontsize=12)

for i_c6, c6_val in enumerate(c6_vals):
    ax2.stairs(h_sbi_c6[i_c6].values() / h_sbi_sm.values(), xbins, color=c6_colors[i_c6], linestyle=c6_lines[i_c6], linewidth=lw)
ax2.stairs(h_sbi_sm.values() / h_sbi_sm.values(), xbins, color=color_sm, linestyle=line_sm, linewidth=lw)

ax1.set_yscale('log')
ax1.set_ylabel('$\\mathrm{d}\sigma / \\mathrm{d} m_{ZZ}\\ \\mathrm{[fb / GeV]}$', loc='top', fontsize=15)
ax1.set_ylim(1e-4, 10.0)
ax1.yaxis.set_ticks([1e-4, 1e-3, 1e-2, 1e-1, 1.0, 10.0])
ax1.yaxis.set_ticklabels(['$10^{-4}$', '$10^{-3}$', '$10^{-2}$', '$0.1$', '$1.0$', '$10$'])

ax1.text(0.04 ,0.12, '$gg (\\rightarrow h^{\\ast}) \\rightarrow ZZ$', transform=ax1.transAxes, ha='left', va='bottom', fontsize=12)
ax1.text(0.04 ,0.04, '$\\sqrt{s} = 14\\,\\mathrm{TeV},\\  3\\, \\mathrm{ab}^{-1}$', transform=ax1.transAxes, ha='left', va='bottom', fontsize=12)

ax2.set_ylabel('$\\mathrm{BSM} / \\mathrm{SM}$', fontsize=15)
ax2.set_ylim(0.95,1.25)

ax2.set_xscale('linear')
ax2.set_xlim(xmin, xmax)
ax2.set_xlabel('$m_{ZZ}\\ \\mathrm{[GeV]}$', loc='right', fontsize=15)
ax2.tick_params(top=True, labeltop=False)
ax2.xaxis.set_ticks([180, 250, 500, 750, 1000])

fig.tight_layout()
fig.subplots_adjust(hspace=0)

fig.canvas.draw()  # update positions
ax1_pos, ax2_pos = ax1.get_position(), ax2.get_position()
ax2.set_position([ax1_pos.x0, ax2_pos.y0, ax1_pos.width, ax2_pos.height]) # align 2nd x-axis with 1st

plt.savefig('sbi_4l_mzz.pdf')
plt.close()

In [42]:
fig, (ax1, ax2) = plt.subplots(2,1, gridspec_kw={'height_ratios': [3,1]}, figsize=(5,5), sharex=True)

l_c6 = []
for i_c6, c6_val in enumerate(c6_vals):
    l_c6.append(ax1.stairs(h_sig_c6[i_c6].values(), xbins, color=c6_colors[i_c6], linestyle=c6_lines[i_c6], linewidth=lw))
l_sm = ax1.stairs(h_sig_sm.values(), xbins, color=color_sm, linestyle=line_sm, linewidth=lw)

l_sm.set_label('$\mathrm{SM}$')
for i_c6, c6_val in enumerate(c6_vals):
    sgn = '-' if c6_val < 0 else '+'
    l_c6[i_c6].set_label(f'$C_{{H}} = {sgn}{abs(c6_val):d}$')
ax1.legend(frameon=False, fontsize=12)

for i_c6, c6_val in enumerate(c6_vals):
    ax2.stairs(h_sig_c6[i_c6].values() / h_sig_sm.values(), xbins, color=c6_colors[i_c6], linestyle=c6_lines[i_c6], linewidth=lw)
ax2.stairs(h_sig_sm.values() / h_sig_sm.values(), xbins, color=color_sm, linestyle=line_sm, linewidth=lw)

ax1.set_yscale('linear')
ax1.set_ylabel('$\\mathrm{d}\sigma / \\mathrm{d} m_{ZZ}\\ \\mathrm{[ab / GeV]}$', loc='top', fontsize=15)
ax1.set_ylim(0.0, 25.0)
ax1.tick_params(labelsize=12)

ax1.text(0.04 ,0.96, '$gg \\rightarrow h^{\\ast} \\rightarrow ZZ$', transform=ax1.transAxes, ha='left', va='top', fontsize=12)
ax1.text(0.04 ,0.88, '$\\sqrt{s} = 14\\,\\mathrm{TeV},\\  3\\, \\mathrm{ab}^{-1}$', transform=ax1.transAxes, ha='left', va='top', fontsize=12)

ax2.set_ylabel('$\\mathrm{BSM} / \\mathrm{SM}$', fontsize=15)
ax2.set_ylim(0.0,2.0)
ax2.yaxis.set_ticks([0.0, 0.5, 1.0, 1.5])

ax2.set_xscale('linear')
ax2.set_xlim(xmin, xmax)
ax2.set_xlabel('$m_{ZZ}\\ \\mathrm{[GeV]}$', loc='right', fontsize=15)
ax2.tick_params(top=True, labeltop=False, labelsize=12)

fig.tight_layout()
fig.subplots_adjust(hspace=0)

fig.canvas.draw()  # update positions
ax1_pos, ax2_pos = ax1.get_position(), ax2.get_position()
ax2.set_position([ax1_pos.x0, ax2_pos.y0, ax1_pos.width, ax2_pos.height]) # align 2nd x-axis with 1st

plt.savefig('sig_4l_mzz.pdf')
plt.close()

In [43]:
xobs_sbi = events_sbi.kinematics['l1_pt'].to_numpy()
xobs_sig = events_sig.kinematics['l1_pt'].to_numpy()

nxbins = 35
xmin, xmax = 20.0, 400.0
xbins = np.linspace(xmin, xmax, nxbins + 1)
xcenters = 0.5 * (xbins[:-1] + xbins[1:])
xwidths = np.diff(xbins)

In [44]:
fb_to_ab = 1000.0
h_sig_sm = hist.Hist(hist.axis.Variable(xbins), storage=hist.storage.Weight())
h_sig_sm.fill(xobs_sig, weight=events_sig.weights * fb_to_ab)

h_sig_c6 = []
for i_c6, c6_val in enumerate(c6_vals):
    h = hist.Hist(hist.axis.Variable(xbins), storage=hist.storage.Weight())
    h.fill(xobs_sig, weight=wt_sig_c6[:,i_c6] * fb_to_ab)
    h_sig_c6.append(h)

h_sbi_sm = hist.Hist(hist.axis.Variable(xbins), storage=hist.storage.Weight())
h_sbi_sm.fill(xobs_sbi, weight=events_sbi.weights)

h_sbi_c6 = []
for i_c6, c6_val in enumerate(c6_vals):
    h = hist.Hist(hist.axis.Variable(xbins), storage=hist.storage.Weight())
    h.fill(xobs_sbi, weight=wt_sbi_c6[:,i_c6])
    h_sbi_c6.append(h)


In [45]:
fig, (ax1, ax2) = plt.subplots(2,1, gridspec_kw={'height_ratios': [3,1]}, figsize=(5,5), sharex=True)

l_c6 = []
for i_c6, c6_val in enumerate(c6_vals):
    l_c6.append(ax1.stairs(h_sbi_c6[i_c6].values(), xbins, color=c6_colors[i_c6], linestyle=c6_lines[i_c6], linewidth=lw))
l_sm = ax1.stairs(h_sbi_sm.values(), xbins, color=color_sm, linestyle=line_sm, linewidth=lw)

l_sm.set_label('$\mathrm{SM}$')
for i_c6, c6_val in enumerate(c6_vals):
    sgn = '-' if c6_val < 0 else '+'
    l_c6[i_c6].set_label(f'$C_{{H}} = {sgn}{abs(c6_val):d}$')
ax1.legend(frameon=False, fontsize=12)

for i_c6, c6_val in enumerate(c6_vals):
    ax2.stairs(h_sbi_c6[i_c6].values() / h_sbi_sm.values(), xbins, color=c6_colors[i_c6], linestyle=c6_lines[i_c6], linewidth=lw)
ax2.stairs(h_sbi_sm.values() / h_sbi_sm.values(), xbins, color=color_sm, linestyle=line_sm, linewidth=lw)

ax1.set_yscale('log')
ax1.set_ylabel('$\\mathrm{d}\sigma / \\mathrm{d} p_{\\mathrm{T}}^{\\ell_1}\\ \\mathrm{[fb / GeV]}$', loc='top', fontsize=15)
ax1.set_ylim(1e-5, 1.0)
ax1.yaxis.set_ticks([1e-5, 1e-4, 1e-3, 1e-2, 1e-1, 1.0])
ax1.yaxis.set_ticklabels(['$10^{-5}$', '$10^{-4}$', '$10^{-3}$', '$10^{-2}$', '$0.1$', '$1.0$'])
ax1.tick_params(labelsize=12)

ax1.text(0.04 ,0.12, '$gg (\\rightarrow h^{\\ast}) \\rightarrow ZZ$', transform=ax1.transAxes, ha='left', va='bottom')
ax1.text(0.04 ,0.04, '$\\sqrt{s} = 14\\,\\mathrm{TeV},\\  3\\, \\mathrm{ab}^{-1}$', transform=ax1.transAxes, ha='left', va='bottom')

ax2.set_ylabel('$\\mathrm{BSM} / \\mathrm{SM}$', fontsize=15)
ax2.set_ylim(0.95,1.25)

ax2.set_xscale('linear')
ax2.set_xlim(xmin, xmax)
ax2.set_xlabel('$p_{\\mathrm{T}}^{\\ell_1}\\ \\mathrm{[GeV]}$', loc='right', fontsize=15)
ax2.xaxis.set_ticks([20.0, 100.0, 200.0, 300.0, 400.0])
ax2.tick_params(top=True, labeltop=False, labelsize=12)
ax2.xaxis.set_ticks([20.0, 100.0, 200.0, 300.0, 400.0])

fig.tight_layout()
fig.subplots_adjust(hspace=0)

fig.canvas.draw()  # update positions
ax1_pos, ax2_pos = ax1.get_position(), ax2.get_position()
ax2.set_position([ax1_pos.x0, ax2_pos.y0, ax1_pos.width, ax2_pos.height]) # align 2nd x-axis with 1st

plt.savefig('sbi_4l_ptl1.pdf')
plt.close()

In [46]:
fig, (ax1, ax2) = plt.subplots(2,1, gridspec_kw={'height_ratios': [3,1]}, figsize=(5,5), sharex=True)

l_c6 = []
for i_c6, c6_val in enumerate(c6_vals):
    l_c6.append(ax1.stairs(h_sig_c6[i_c6].values(), xbins, color=c6_colors[i_c6], linestyle=c6_lines[i_c6], linewidth=lw))
l_sm = ax1.stairs(h_sig_sm.values(), xbins, color=color_sm, linestyle=line_sm, linewidth=lw)

l_sm.set_label('$\mathrm{SM}$')
for i_c6, c6_val in enumerate(c6_vals):
    sgn = '-' if c6_val < 0 else '+'
    l_c6[i_c6].set_label(f'$C_{{H}} = {sgn}{abs(c6_val):d}$')
ax1.legend(frameon=False, fontsize=12)

for i_c6, c6_val in enumerate(c6_vals):
    ax2.stairs(h_sig_c6[i_c6].values() / h_sig_sm.values(), xbins, color=c6_colors[i_c6], linestyle=c6_lines[i_c6], linewidth=lw)
ax2.stairs(h_sig_sm.values() / h_sig_sm.values(), xbins, color=color_sm, linestyle=line_sm, linewidth=lw)

ax1.set_yscale('linear')
ax1.set_ylabel('$\\mathrm{d}\sigma / \\mathrm{d} p_{\\mathrm{T}}^{\\ell_1}\\ \\mathrm{[ab / GeV]}$', loc='top', fontsize=15)
ax1.set_ylim(0.0, 30.0)
ax1.tick_params(labelsize=12)

ax1.text(0.04 ,0.96, '$gg \\rightarrow h^{\\ast} \\rightarrow ZZ$', transform=ax1.transAxes, ha='left', va='top', fontsize=12)
ax1.text(0.04 ,0.88, '$\\sqrt{s} = 14\\,\\mathrm{TeV},\\  3\\, \\mathrm{ab}^{-1}$', transform=ax1.transAxes, ha='left', va='top', fontsize=12)

ax2.set_ylabel('$\\mathrm{BSM} / \\mathrm{SM}$', fontsize=15)
ax2.set_ylim(0.0,2.0)
ax2.yaxis.set_ticks([0.0, 0.5, 1.0, 1.5])

ax2.set_xscale('linear')
ax2.set_xlim(xmin, xmax)
ax2.set_xlabel('$p_{\\mathrm{T}}^{\\ell_1}\\ \\mathrm{[GeV]}$', loc='right', fontsize=15)
ax2.tick_params(top=True, labeltop=False, labelsize=12)
ax2.xaxis.set_ticks([20.0, 100.0, 200.0, 300.0, 400.0])

fig.tight_layout()
fig.subplots_adjust(hspace=0)

fig.canvas.draw()  # update positions
ax1_pos, ax2_pos = ax1.get_position(), ax2.get_position()
ax2.set_position([ax1_pos.x0, ax2_pos.y0, ax1_pos.width, ax2_pos.height]) # align 2nd x-axis with 1st

plt.savefig('sig_4l_ptl1.pdf')
plt.close()